# STAT 764 · Meeting 1 — Predict this

**Tuesday, September 8 · Studio block**

You do not need anything installed today. This notebook runs in your browser.
We will set up Python on your own machine on Thursday.

---

## The task

You have sale records for **120 houses in Ames, Iowa**. Each row is one sale:
square footage, year built, number of bedrooms, neighborhood, and so on — plus
what the house actually sold for.

Somewhere else, I have **250 more houses** that you have never seen. You will
get everything about them *except* the price.

**Predict those 250 prices. Use any method you know.** Linear regression is
fine. So is a tree, a nearest-neighbor rule, or a formula you make up. There is
no grade attached to this and no wrong method.

Then answer one question, in writing, before you see the answer:

> **How well do you think you did?**

In [ ]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026/main/course/data/"

train = pd.read_csv(URL + "ames_day1.csv")
holdout = pd.read_csv(URL + "ames_day1_holdout.csv")

print(f"train:    {train.shape[0]} houses, {train.shape[1]} columns")
print(f"holdout:  {holdout.shape[0]} houses, {holdout.shape[1]} columns (no SalePrice)")
train.head()

## What's in it

| column | meaning |
|---|---|
| `Gr_Liv_Area` | above-ground living area, sq ft |
| `Lot_Area` | lot size, sq ft |
| `Year_Built` | year of construction |
| `Overall_Qual` | overall material and finish quality, 1–10 |
| `Overall_Cond` | overall condition, 1–10 |
| `Total_Bsmt_SF` | total basement area, sq ft |
| `Full_Bath`, `Half_Bath` | bathrooms above grade |
| `Bedroom_AbvGr` | bedrooms above grade |
| `TotRms_AbvGrd` | total rooms above grade |
| `Garage_Cars` | garage capacity, cars |
| `Fireplaces` | number of fireplaces |
| `Neighborhood` | one of 28 Ames neighborhoods |
| `Central_Air` | `Y` or `N` |
| `SalePrice` | **what it sold for — this is what you predict** |

In [ ]:
train["SalePrice"].describe()

## If you are new to Python

You are a statistician who knows R. Here is the whole phrasebook you need today.

| you want to | in R | here |
|---|---|---|
| a column | `df$SalePrice` | `df["SalePrice"]` |
| drop a column | `df[, -1]` | `df.drop(columns="SalePrice")` |
| dummy-code factors | happens automatically | `pd.get_dummies(df)` |
| fit a linear model | `lm(y ~ ., data=d)` | `LinearRegression().fit(X, y)` |
| predict | `predict(m, newdata)` | `m.predict(X_new)` |
| R-squared | `summary(m)$r.squared` | `r2_score(y, m.predict(X))` |

The one real difference: in Python you hand the model **two** objects — a table
of predictors `X` and a vector of outcomes `y` — instead of a formula and a data
frame. Everything else is a renaming.

## A baseline that already works

Run this cell. It predicts the same number — the average price — for every
house. It is a terrible model, and it is the number every other model has to
beat. **Every problem in this course starts with a baseline.**

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error

baseline_prediction = train["SalePrice"].mean()
baseline_guesses = np.full(len(train), baseline_prediction)

print(f"Predict ${baseline_prediction:,.0f} for every house.")
print(f"  R-squared: {r2_score(train['SalePrice'], baseline_guesses):.3f}")
print(f"  RMSE:      ${root_mean_squared_error(train['SalePrice'], baseline_guesses):,.0f}")

An R-squared of 0.000, by construction. Now beat it.

## Your turn

Build something better. These are all available — import what you want:

```python
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
```

`pd.get_dummies(...)` will turn `Neighborhood` and `Central_Air` into numbers
for you. Do not spend more than about fifteen minutes here.

In [ ]:
X = pd.get_dummies(train.drop(columns="SalePrice"))
y = train["SalePrice"]

# YOUR CODE HERE — fit a model, then score it however you normally would.
#
# my_model = ...
# my_model.fit(X, y)

---

## Now commit to a number

**Before you run anything else.** Fill in your honest expectation for the 250
houses you have never seen, and say what your number is based on.

This is the actual point of the exercise, so do it before you scroll further.

In [ ]:
MY_ESTIMATE = None        # your expected R-squared on the 250 unseen houses
HOW_I_GOT_IT = ""         # one sentence: where did that number come from?

assert MY_ESTIMATE is not None, "Write down a number before you go on."
assert HOW_I_GOT_IT, "Say where the number came from."
print(f"On the record: R-squared of about {MY_ESTIMATE}, because {HOW_I_GOT_IT}")

## Predict the 250

In [ ]:
# Line up the holdout columns with your training columns.
X_holdout = pd.get_dummies(holdout.drop(columns="id")).reindex(columns=X.columns, fill_value=0)

my_predictions = my_model.predict(X_holdout)      # <- your model from above

print(f"{len(my_predictions)} predictions, averaging ${my_predictions.mean():,.0f}")

## The answer

In [ ]:
key = pd.read_csv(URL + "ames_day1_key.csv")
truth = key.set_index("id").loc[holdout["id"], "SalePrice"].to_numpy()

actual_r2 = r2_score(truth, my_predictions)
actual_rmse = root_mean_squared_error(truth, my_predictions)

print(f"  You said:   R-squared about {MY_ESTIMATE}")
print(f"  You got:    R-squared {actual_r2:.3f}   (RMSE ${actual_rmse:,.0f})")
print()
gap = MY_ESTIMATE - actual_r2
if gap > 0.03:
    print(f"  You were optimistic by {gap:.3f}.")
elif gap < -0.03:
    print(f"  You were pessimistic by {-gap:.3f}.")
else:
    print("  You called it.")

---

## Compare

Put two numbers on the board: **what you said**, and **what you got**.

Then we look at the room together:

1. Whose two numbers were furthest apart? What method were they using?
2. Did anyone report an R-squared of 1.000 for their model? What does that mean?
3. Is the person with the **highest first number** the same as the person with
   the **highest second number**?
4. Where did your first number come from? If you scored your model on the same
   120 houses you fit it to — what exactly did that number measure?

## Exit ticket

Two sentences, submitted before you leave:

> **You reported a number you could not have known was right. What would you
> need to do differently to produce a number you could stand behind?**

---

## Before Thursday

Install Python and VS Code following the **Software Setup** page on Canvas. It
takes about twenty minutes. Reply to my email when `check_setup.py` prints
"All good" — if it does not, send me what it printed and we will sort it out
before class rather than during it.

**Lab 1 is assigned today, due Tuesday Sep 15 at 11:59 pm.** You will need
Thursday's material to finish it, so read it now and start it after Thursday.

---

## Also in this neighbourhood

Every meeting ends with this section: things next to today's material that we
are not doing, and what kind of "not doing" each one is.

| tag | meaning |
|---|---|
| 📗 **On your own** | You have the tools to read this now. Worth your time. |
| 🎓 **Next course** | Real material, needs more room than we have. |
| 🚫 **Not used, and why** | A method you may already know that we deliberately avoid. |

---

📗 **ISLP §2.1–2.2.** The formal version of today: the setup, and why some error
is *irreducible* no matter what model you use. Free PDF at
[statlearning.com](https://www.statlearning.com/).

🚫 **R-squared as a way to choose between models.** You already know it, and it
is the single most common way people do what happened in this room today. Adding
any predictor — including a column of random noise — can only increase R-squared
on the data you fit to. Adjusted R-squared patches that with a penalty, but it is
still computed on the training data and still is not an estimate of how you will
do on a new house. We use held-out error instead, all semester.